# Treinamento com interface de alto nível

## Importação das bibliotecas

In [1]:
# http://pytorch.org/
from os.path import exists

import torch

In [2]:
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR

## Criação da rede

In [3]:
input_size = 32*32
input_size

1024

In [4]:
class CifarNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(input_size, 512)
    self.fc2 = nn.Linear(512, 256)
    self.fc3 = nn.Linear(256, 128)
    self.fc4 = nn.Linear(128, 64)
    self.fc5 = nn.Linear(64, 10)

  def forward(self, x):
    x = x.view(x.shape[0], -1)
    x = self.fc1(x)
    x = F.relu(x)
    x = self.fc2(x)
    x = F.relu(x)
    x = self.fc3(x)
    x = F.relu(x)
    x = self.fc4(x)
    x = F.relu(x)
    x = self.fc5(x)
    output = F.log_softmax(x, dim=1)
    return output

model = CifarNet()
model

CifarNet(
  (fc1): Linear(in_features=1024, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=128, bias=True)
  (fc4): Linear(in_features=128, out_features=64, bias=True)
  (fc5): Linear(in_features=64, out_features=10, bias=True)
)

In [6]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Grayscale(num_output_channels=1),
    transforms.Normalize((0.1307,), (0.3081,))
    ])
dataset_train = datasets.CIFAR10('../data', train=True, download=True, transform=transform)
train_kwargs = {'batch_size': 64}


train_loader = torch.utils.data.DataLoader(dataset_train,**train_kwargs)

100%|██████████| 170498071/170498071 [00:13<00:00, 12914877.34it/s]


Extracting ../data/cifar-10-python.tar.gz to ../data


In [7]:
for batch_idx, (data, target) in enumerate(train_loader):
    data, target = data, target
    print(data.shape)
    aux = data.view(data.shape[0], -1)
    print(aux.shape)

torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
torch.Size([64, 1024])
torch.Size([64, 1, 32, 32])
tor

## Treinamento

### Criando o objeto de treinamento

In [9]:
def train(log_interval, dry_run, model, device, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
            if dry_run:
                break

In [10]:
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))

## Avaliação

In [11]:
use_cuda = torch.cuda.is_available()

torch.manual_seed(1111)

device = torch.device("cuda" if use_cuda else "cpu")

train_kwargs = {'batch_size': 500}
test_kwargs = {'batch_size': 100}
if use_cuda:
    cuda_kwargs = {'num_workers': 1,
                    'pin_memory': True,
                    'shuffle': True}
    train_kwargs.update(cuda_kwargs)
    test_kwargs.update(cuda_kwargs)

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Grayscale(num_output_channels=1),
    transforms.Normalize((0.1307,), (0.3081,))
    ])
dataset_train = datasets.CIFAR10('../data', train=True, download=True,
                    transform=transform)
dataset_test = datasets.CIFAR10('../data', train=False, download=True,
                    transform=transform)
train_loader = torch.utils.data.DataLoader(dataset_train,**train_kwargs)
test_loader = torch.utils.data.DataLoader(dataset_test, **test_kwargs)

model = CifarNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 14
scheduler = StepLR(optimizer, step_size=4, gamma=0.7)

for epoch in range(1, epochs + 1):
    train(100, False, model, device, train_loader, optimizer, epoch)
    test(model, device, test_loader)
    scheduler.step()

torch.save(model.state_dict(), "mnist_cnn.pt")

Files already downloaded and verified
Files already downloaded and verified
Train Epoch: 1 [0/50000 (0%)]	Loss: 2.304829

Test set: Average loss: 1.9890, Accuracy: 2736/10000 (27%)

Train Epoch: 2 [0/50000 (0%)]	Loss: 2.020265

Test set: Average loss: 1.8524, Accuracy: 3426/10000 (34%)

Train Epoch: 3 [0/50000 (0%)]	Loss: 1.910685

Test set: Average loss: 1.7824, Accuracy: 3678/10000 (37%)

Train Epoch: 4 [0/50000 (0%)]	Loss: 1.780693

Test set: Average loss: 1.7521, Accuracy: 3728/10000 (37%)

Train Epoch: 5 [0/50000 (0%)]	Loss: 1.774840

Test set: Average loss: 1.7100, Accuracy: 3883/10000 (39%)

Train Epoch: 6 [0/50000 (0%)]	Loss: 1.678938

Test set: Average loss: 1.6772, Accuracy: 4023/10000 (40%)

Train Epoch: 7 [0/50000 (0%)]	Loss: 1.634197

Test set: Average loss: 1.6398, Accuracy: 4156/10000 (42%)

Train Epoch: 8 [0/50000 (0%)]	Loss: 1.558869

Test set: Average loss: 1.6436, Accuracy: 4132/10000 (41%)

Train Epoch: 9 [0/50000 (0%)]	Loss: 1.564933

Test set: Average loss: 1.6361